# NB_lab_test_trends

Builds `adb_caresync.gold.lab_test_trends` — one row per test type per calendar month with test volumes, abnormal/critical rates, and month-over-month trend. Full rebuild via `CREATE OR REPLACE TABLE`.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends AS
WITH monthly_agg AS (
    SELECT
        lr.test_name,
        YEAR(lr.test_date) AS year,
        MONTH(lr.test_date) AS month,
        COUNT(*) AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END) AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END) AS critical_test_count
    FROM adb_caresync.silver.lab_results lr
    GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
)
SELECT
    ma.test_name,
    ma.year,
    ma.month,
    c.month_name,
    c.quarter_name,
    ma.total_tests,
    ma.abnormal_test_count,
    ma.critical_test_count,
    CAST(ROUND(ma.abnormal_test_count / ma.total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(ma.critical_test_count / ma.total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
    CAST(LAG(ROUND(ma.abnormal_test_count / ma.total_tests * 100, 2))
        OVER (PARTITION BY ma.test_name ORDER BY ma.year, ma.month) AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
    CAST(ROUND(
        ROUND(ma.abnormal_test_count / ma.total_tests * 100, 2)
        - LAG(ROUND(ma.abnormal_test_count / ma.total_tests * 100, 2))
            OVER (PARTITION BY ma.test_name ORDER BY ma.year, ma.month),
        2
    ) AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM monthly_agg ma
LEFT JOIN adb_caresync.silver.calendar c
    ON c.date_key = CAST(DATE_FORMAT(MAKE_DATE(ma.year, ma.month, 1), 'yyyyMMdd') AS INT)

In [0]:
updated_df = spark.table("adb_caresync.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))